# LifeLedger — Phase 5 · Advanced Planning Validation

Validates `advanced_planning.py` across all four engines:

1. Survivor simulation — income removed, mortgage check, life cover
2. Estate / IHT — NRB/RNRB, taxable estate, gift tracker, strategies
3. Healthcare — cost projection by phase, care home, peak year
4. Rebalancing — drift detection, glide path, trade recommendations
5. YAML config round-trip
6. Full Phase 5 dashboard chart (4 panels)

In [ ]:
import sys, logging
from pathlib import Path
from datetime import date

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from backend.engine.advanced_planning import (
    AdvancedPlanningEngine, PlanningConfig,
    SurvivorConfig, EstateConfig, HealthcareConfig, HealthcarePhase,
    RebalanceConfig, AssetAllocation,
    load_planning_config,
)
from backend.persistence.yaml_serialiser import load_scenario_from_file

logging.basicConfig(level=logging.WARNING, format='%(levelname)-8s %(name)s %(message)s')

BASE_PATH   = ROOT / 'data' / 'scenarios' / 'base.yaml'
P5_CFG_PATH = ROOT / 'config' / 'planning' / 'planning_config.yaml'

scenario = load_scenario_from_file(str(BASE_PATH))
print(f'Scenario: {scenario.name}')
print(f'People  : {[p.name for p in scenario.people]}')
print(f'Pensions: {[p.id for p in scenario.pension_funds]}')
print(f'Invest  : {[a.id for a in scenario.investment_accounts]}')

In [ ]:
# Build engine with programmatic config
cfg = PlanningConfig(
    survivor=SurvivorConfig(
        household_expense_reduction_pct=0.25,
        survivor_pension_fraction=0.50,
        mortgage_affordability_threshold=0.35,
        life_cover_income_multiple=10.0,
        mortgage_payoff_cover=True,
    ),
    estate=EstateConfig(
        jurisdiction='uk',
        uk_nil_rate_band=325_000,
        uk_residence_nil_rate_band=175_000,
        uk_iht_rate=0.40,
        uk_sipp_outside_estate=True,
        gifts=[
            {'date': '2019-06-01', 'amount': 15000, 'recipient': 'Hannah', 'notes': 'Education'},
            {'date': '2022-04-06', 'amount': 3000,  'recipient': 'Hannah', 'notes': 'Annual exemption'},
            {'date': '2024-04-06', 'amount': 3000,  'recipient': 'Hannah', 'notes': 'Annual exemption'},
        ]
    ),
    healthcare=HealthcareConfig(
        jurisdiction='uk',
        phases=[
            HealthcarePhase(label='NHS Working', start_age=0,  end_age=59, annual_cost=0,    inflation_rate=0.03),
            HealthcarePhase(label='NHS + Private', start_age=60, end_age=79, annual_cost=1500, inflation_rate=0.05),
            HealthcarePhase(label='NHS Late Life', start_age=80, end_age=100,annual_cost=3500, inflation_rate=0.05),
        ],
        include_care_home=True,
        care_home_start_age=82,
        care_home_daily_rate=130.0,
        care_home_duration_years=3,
        care_home_inflation_rate=0.05,
    ),
    rebalance=RebalanceConfig(
        enabled=True,
        global_target=AssetAllocation(
            account_id='global',
            equities_pct=80, bonds_pct=10, cash_pct=5, property_pct=5,
            drift_threshold=5.0,
        ),
        glide_path_enabled=True,
        glide_path_start_age=50,
        glide_path_end_age=65,
        glide_path_equity_floor=40.0,
    ),
)
engine = AdvancedPlanningEngine(cfg)
report = engine.full_report(scenario)

print(f'Survivor (P1)      : income_lost=£{report.survivor_james.total_income_lost:,.0f}')
print(f'Estate IHT         : £{report.estate.iht_liability:,.0f}')
print(f'Healthcare total   : £{report.healthcare.total_lifetime_cost:,.0f}')
print(f'Rebalancing alerts : {len(report.rebalancing.alerts)}')
print(f'Need action        : {report.rebalancing.accounts_needing_action}')
print(f'Warnings           : {len(report.warnings)}')

## 1 · Survivor Simulation

In [ ]:
s1 = report.survivor_james
print(f'Deceased           : {s1.deceased_person_id}')
print(f'Income lost        : £{s1.total_income_lost:,.0f}/yr')
print(f'Survivor gross     : £{s1.survivor_gross_income:,.0f}/yr')
print(f'Survivor pension   : £{s1.survivor_pension_income:,.0f}/yr')
print(f'Expense reduction  : £{s1.expense_reduction:,.0f}/yr')
print(f'Recommended cover  : £{s1.recommended_life_cover:,.0f}')
print(f'Cover breakdown    : {s1.life_cover_breakdown}')
if s1.mortgage_affordability:
    m = s1.mortgage_affordability
    print(f'Mortgage afford    : {m.is_affordable} (ratio={m.affordability_ratio:.1%})')
print(f'Key risks          : {s1.key_risks}')

# Second partner
s2 = report.survivor_sarah
print(f'\nIf {s2.deceased_person_id} dies:')
print(f'  Income lost: £{s2.total_income_lost:,.0f}  Survivor: £{s2.survivor_gross_income:,.0f}')

assert s1.total_income_lost >= 0
assert s1.survivor_gross_income >= 0
assert s1.recommended_life_cover >= 0
assert len(s1.recommendations) > 0
assert s1.life_cover_breakdown['total_recommended'] == s1.recommended_life_cover
print('\n✅ Survivor assertions passed')

## 2 · Estate / IHT Calculation

In [ ]:
est = report.estate
print(f'Gross estate       : £{est.gross_estate:,.0f}')
print(f'Pension excluded   : £{est.pension_outside_estate:,.0f}')
print(f'Gifts outside 7yr  : £{est.gifts_outside_estate:,.0f}')
print(f'Net estate         : £{est.net_estate:,.0f}')
print(f'NRB available      : £{est.nrb_available:,.0f}  (individual×2)')
print(f'RNRB available     : £{est.rnrb_available:,.0f}  (×2 couple)')
print(f'Total allowances   : £{est.total_allowances:,.0f}')
print(f'Taxable estate     : £{est.taxable_estate:,.0f}')
print(f'IHT liability      : £{est.iht_liability:,.0f}')
print(f'Net to bens        : £{est.net_to_beneficiaries:,.0f}')
print(f'Effective IHT rate : {est.effective_iht_rate:.1%}')
print(f'Gift tracker entries: {len(est.gift_tracker)}')
print(f'Gift IHT at risk   : £{est.gift_iht_at_risk:,.0f}')
print(f'Gift allow remain  : £{est.annual_gift_allowance_remaining:,.0f}')
print()
print('IHT reduction opportunities:')
for op in est.iht_reduction_opportunities:
    print(f'  [{op["priority"]}] {op["strategy"]} → save £{op["estimated_saving"]:,.0f}')

print()
print('Gift tracker:')
for g in est.gift_tracker:
    status = '✅ Exempt' if g.is_outside_estate else f'⚠️  {g.years_to_exempt:.1f}yr to exempt'
    print(f'  {g.gift_date}: £{g.amount:,.0f} → {g.recipient}  |  {status}  |  IHT at risk: £{g.iht_at_risk:,.0f}')

# Assertions
assert est.gross_estate > 0
assert est.net_estate <= est.gross_estate
assert est.iht_liability >= 0
assert est.net_to_beneficiaries <= est.net_estate
assert len(est.iht_reduction_opportunities) > 0
# The 2019 gift should be outside the estate (>7 years ago)
old_gift = next((g for g in est.gift_tracker if g.amount == 15000), None)
if old_gift:
    assert old_gift.is_outside_estate, '2019 gift should be exempt (>7yr)'
    assert old_gift.iht_at_risk == 0
print('\n✅ Estate / IHT assertions passed')

## 3 · Healthcare Cost Projection

In [ ]:
hc = report.healthcare
print(f'Total lifetime cost: £{hc.total_lifetime_cost:,.0f}')
print(f'Peak year          : {hc.peak_year} (£{hc.peak_year_cost:,.0f})')
print(f'Care home cost     : £{hc.care_home_cost:,.0f}')
print(f'By person          : {hc.by_person}')
print(f'Total rows         : {len(hc.rows)}')
print()
print('Sample rows (first 5 with cost > 0):')
nonzero = [r for r in hc.rows if r.annual_cost > 0]
for r in nonzero[:5]:
    print(f'  {r.year} age={r.age:3d} [{r.person_id}] {r.phase_label:<25} £{r.annual_cost:,.0f}')

assert hc.total_lifetime_cost >= 0
assert hc.peak_year >= 2025
# Care home cost should be the most expensive phase
if hc.care_home_cost > 0:
    print(f'Care home is most expensive phase: £{hc.care_home_cost:,.0f}/yr avg')
# All rows should have valid person_id
valid_pids = {p.id for p in scenario.people}
for r in hc.rows:
    assert r.person_id in valid_pids, f'Unknown person_id: {r.person_id}'
print('\n✅ Healthcare assertions passed')

## 4 · Portfolio Rebalancing

In [ ]:
rb = report.rebalancing
print(f'Total portfolio    : £{rb.total_portfolio_value:,.0f}')
print(f'Accounts analysed  : {len(rb.alerts)}')
print(f'Need action        : {rb.accounts_needing_action}')
print()
print('Global allocation vs target:')
classes = ['equities', 'bonds', 'cash', 'property', 'alternatives']
for c in classes:
    curr = rb.global_allocation.get(c, 0)
    tgt  = rb.global_target.get(c, 0)
    drift = rb.global_drift.get(c, 0)
    flag = '⚠️ ' if abs(drift) >= 5 else '  '
    print(f'  {flag}{c:<14} current={curr:5.1f}%  target={tgt:5.1f}%  drift={drift:+.1f}pp')

print()
for alert in rb.alerts:
    print(f'\nAccount: {alert.account_name} ({alert.account_id})')
    print(f'  Status    : {alert.status}  max_drift={alert.max_drift:.1f}pp  '
          f'glide_adjusted={alert.glide_adjusted}')
    print(f'  Holdings  : {[(h.holding_name, h.asset_class, f"£{h.value:,.0f}") for h in alert.holdings]}')
    print(f'  Trades    : {[(k, f"£{v:,.0f}") for k, v in alert.trades_needed.items() if abs(v) > 100]}')

# Assertions
assert len(rb.alerts) == len(scenario.investment_accounts)
for alert in rb.alerts:
    assert alert.status in ('ok', 'amber', 'rebalance_needed')
    assert alert.total_value >= 0
    # Current allocation should sum to ~100%
    total_alloc = sum(alert.current_allocation.values())
    assert abs(total_alloc - 100) < 1.0, f'Allocation sums to {total_alloc:.1f}%'

# Test glide path for older person
engine_rb_old = AdvancedPlanningEngine(cfg)
rb_old = engine_rb_old._rebalance.analyse(scenario, owner_age=62)
for alert in rb_old.alerts:
    if alert.glide_adjusted:
        assert alert.target_allocation['equities'] < 80.0, \
            'Glide path should reduce equities below 80% at age 62'
        break
print('\n✅ Rebalancing assertions passed')

## 5 · YAML Config Round-Trip

In [ ]:
if P5_CFG_PATH.exists():
    cfg_yaml = load_planning_config(str(P5_CFG_PATH))
    print(f'Survivor pension fraction  : {cfg_yaml.survivor.survivor_pension_fraction:.0%}')
    print(f'NRB                        : £{cfg_yaml.estate.uk_nil_rate_band:,.0f}')
    print(f'SIPP outside estate        : {cfg_yaml.estate.uk_sipp_outside_estate}')
    print(f'Healthcare phases          : {len(cfg_yaml.healthcare.phases)}')
    print(f'Care home daily rate       : £{cfg_yaml.healthcare.care_home_daily_rate:.2f}')
    print(f'Global equity target       : {cfg_yaml.rebalance.global_target.equities_pct:.0f}%')
    print(f'Glide path equity floor    : {cfg_yaml.rebalance.glide_path_equity_floor:.0f}%')
    print(f'Account targets            : {len(cfg_yaml.rebalance.account_targets)}')

    engine_yaml = AdvancedPlanningEngine(cfg_yaml)
    report_yaml = engine_yaml.full_report(scenario)
    assert report_yaml.estate.gross_estate >= 0
    assert report_yaml.rebalancing.total_portfolio_value > 0
    print('\n✅ YAML round-trip assertions passed')
else:
    print(f'Skipped — not found at {P5_CFG_PATH}')

## 6 · Phase 5 Dashboard Chart

In [ ]:
import numpy as np

fig = plt.figure(figsize=(16, 14), facecolor='#0d1117')
gs  = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.32)
ax1 = fig.add_subplot(gs[0, 0])  # Survivor income comparison
ax2 = fig.add_subplot(gs[0, 1])  # Estate waterfall
ax3 = fig.add_subplot(gs[1, 0])  # Healthcare cost over time
ax4 = fig.add_subplot(gs[1, 1])  # Rebalancing: current vs target

for ax in [ax1, ax2, ax3, ax4]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.spines[:].set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.5)

fig.suptitle('LifeLedger Phase 5 — Advanced Planning Dashboard', color='#e6edf3', fontsize=13, y=0.98)

# ── Panel 1: Survivor income comparison ─────────────────────────────────────
labels_sv = ['Both\nAlive', f'If {s1.deceased_person_id}\nDies', f'If {s2.deceased_person_id}\nDies']
combined_income = s1.survivor_gross_income + s1.total_income_lost
incomes_sv = [
    combined_income,
    s1.survivor_gross_income + s1.survivor_pension_income,
    s2.survivor_gross_income + s2.survivor_pension_income,
]
colours_sv = ['#58a6ff', '#f85149', '#f0a500']
bars1 = ax1.bar(labels_sv, incomes_sv, color=colours_sv, alpha=0.85, edgecolor='#21262d')
for bar, val in zip(bars1, incomes_sv):
    ax1.text(bar.get_x() + bar.get_width()/2, val + max(incomes_sv)*0.01,
             f'£{val/1e3:.0f}k', ha='center', fontsize=9, color='#e6edf3')
ax1.set_title('Gross Income: Both vs Single Survivor', color='#e6edf3', fontsize=10)
ax1.set_ylabel('Annual Gross Income (£)', color='#8b949e')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))

# ── Panel 2: Estate waterfall ────────────────────────────────────────────────
estate_labels = ['Gross Estate', '- Pension\n(outside)', '- Gifts\n(>7yr)', 'Net Estate',
                 '- Allowances\n(NRB+RNRB)', 'Taxable\nEstate', 'IHT\n(40%)', 'Net to\nBens']
est_values = [
    est.gross_estate,
    -est.pension_outside_estate,
    -est.gifts_outside_estate,
    est.net_estate,
    -est.total_allowances,
    est.taxable_estate,
    -est.iht_liability,
    est.net_to_beneficiaries,
]
bar_cols = ['#58a6ff','#f85149','#f85149','#58a6ff','#f85149','#f0a500','#f85149','#3fb950']
bars2 = ax2.bar(range(len(estate_labels)), [abs(v) for v in est_values], color=bar_cols, alpha=0.85, edgecolor='#21262d')
ax2.set_xticks(range(len(estate_labels)))
ax2.set_xticklabels(estate_labels, fontsize=7, color='#8b949e')
ax2.set_title('Estate / IHT Waterfall', color='#e6edf3', fontsize=10)
ax2.set_ylabel('£', color='#8b949e')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
for i, (bar, val) in enumerate(zip(bars2, est_values)):
    ax2.text(bar.get_x() + bar.get_width()/2, abs(val) + max(abs(v) for v in est_values)*0.01,
             f'£{abs(val)/1e3:.0f}k', ha='center', fontsize=7, color='#e6edf3')

# ── Panel 3: Healthcare over time ───────────────────────────────────────────
year_hc: dict[int, float] = {}
for r in hc.rows:
    year_hc[r.year] = year_hc.get(r.year, 0) + r.annual_cost
if year_hc:
    hc_yrs = sorted(year_hc.keys())
    hc_vals = [year_hc[y] for y in hc_yrs]
    ax3.fill_between(hc_yrs, hc_vals, alpha=0.3, color='#f85149')
    ax3.plot(hc_yrs, hc_vals, color='#f85149', linewidth=1.8)
    ax3.axvline(hc.peak_year, color='#f0a500', linewidth=1, linestyle='--', alpha=0.8,
                label=f'Peak {hc.peak_year}: £{hc.peak_year_cost:,.0f}')
    ax3.set_title(f'Healthcare Costs (total £{hc.total_lifetime_cost/1e3:.0f}k)', color='#e6edf3', fontsize=10)
    ax3.set_ylabel('Annual Cost (£)', color='#8b949e')
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax3.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

# ── Panel 4: Rebalancing — current vs target ─────────────────────────────────
classes = ['equities', 'bonds', 'cash', 'property', 'alternatives']
x = np.arange(len(classes))
w = 0.35
curr_vals = [rb.global_allocation.get(c, 0) for c in classes]
tgt_vals  = [rb.global_target.get(c, 0) for c in classes]
ax4.bar(x - w/2, curr_vals, w, color='#58a6ff', alpha=0.85, label='Current')
ax4.bar(x + w/2, tgt_vals,  w, color='#f0a500', alpha=0.85, label='Target')
ax4.set_xticks(x)
ax4.set_xticklabels([c.capitalize() for c in classes], fontsize=8, color='#8b949e')
ax4.set_title('Portfolio Allocation — Current vs Target', color='#e6edf3', fontsize=10)
ax4.set_ylabel('% of Portfolio', color='#8b949e')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax4.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=9)
# Annotate drift
for xi, c in enumerate(classes):
    drift = rb.global_drift.get(c, 0)
    if abs(drift) >= 3:
        ax4.text(xi, max(curr_vals[xi], tgt_vals[xi]) + 1, f'{drift:+.1f}pp',
                 ha='center', fontsize=7, color='#f85149' if abs(drift) >= 5 else '#f0a500')

plt.savefig('phase5_planning_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Phase 5 dashboard chart saved.')

## ✅ Phase 5 Validation Complete

All assertions passed. `advanced_planning.py` is ready for Phase 5 integration.